## Data Loading

Data Description:  
- 4000 grayscale images of 28x28, vectorized into 784-dimensional feature vectors  
- 2 independent training/test sets, each containing 2000 samples  
- Label range: 0~9 (10 classes of handwritten digits)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import os
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

os.makedirs('result', exist_ok=True)

# Select device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device used: {device}')
print(f'PyTorch version: {torch.__version__}')

# Set random seed for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
DATA_DIR = 'digits4000_txt'

X_all    = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_digits_vec.txt'),    delimiter='\t')
y_all    = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_digits_labels.txt'), dtype=int)
trainset = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_trainset.txt'),      delimiter='\t', dtype=int)
testset  = np.loadtxt(os.path.join(DATA_DIR, 'digits4000_testset.txt'),       delimiter='\t', dtype=int)

# Normalize to [0, 1]
X_norm = (X_all / 255.0).astype(np.float32)

# 2 independent experiment indices (0-based)
exp_idx = {
    1: {'train': trainset[:, 0] - 1, 'test': testset[:, 0] - 1},
    2: {'train': trainset[:, 1] - 1, 'test': testset[:, 1] - 1},
}

print(f'X_all: {X_all.shape}, y_all: {y_all.shape}')
for exp in [1, 2]:
    tr_idx = exp_idx[exp]['train']
    te_idx = exp_idx[exp]['test']
    print(f'Experiment {exp}: Train={len(tr_idx)}, Test={len(te_idx)}, '
          f'Class distribution={dict(zip(*np.unique(y_all[tr_idx], return_counts=True)))}')
print('Data loaded.')

# Handwritten Digit Classification - Machine Learning Methods

Implementation of machine learning classification methods, including:
1. **1-NN**: Baseline
2. **Multinomial Logistic Regression**: Uses statistical methods to estimate class probabilities
3. **SVM (Linear & RBF)**: Uses different kernel functions (linear and radial basis function) to handle non-linear classification problems, adopting a 1-vs-all strategy for the 10 digit classes

2 independent experiments, hyperparameter tuning based on the training set

## 1. PCA Dimensionality Reduction
Compress 784-dimensional pixels to a smaller dimension while retaining 95% of the variance

In [ ]:
from sklearn.decomposition import PCA
def preprocess_with_pca(X_train, X_test, variance_ratio=0.95):
    """
    Perform PCA dimensionality reduction
    """
    pca = PCA(n_components=variance_ratio, svd_solver='full')
    X_train_pca = pca.fit_transform(X_train)
    X_test_pca = pca.transform(X_test)
    
    print(f"PCA completed: Features compressed from 784 dims to {X_train_pca.shape[1]} dims (retained {variance_ratio*100}% variance)")
    return X_train_pca, X_test_pca

## 2. Untuned Baseline

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix

# Define models
models = {
    '1-NN Baseline': KNeighborsClassifier(n_neighbors=1),
    'Logistic Regression': LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000),
    'SVM (Linear)': SVC(kernel='linear', decision_function_shape='ovr'), 
    'SVM (RBF)': SVC(kernel='rbf', decision_function_shape='ovr') 
}

# Store experimental results
results = {name: [] for name in models.keys()}
conf_matrices = {name: [] for name in models.keys()}

# Execute 2-Trial Protocol
for exp_id in [1, 2]:
    print(f"\n--- Running Experiment {exp_id} ---")
    
    tr_idx = exp_idx[exp_id]['train']
    te_idx = exp_idx[exp_id]['test']
    
    X_train, X_test = X_norm[tr_idx], X_norm[te_idx]
    y_train, y_test = y_all[tr_idx], y_all[te_idx]

    # PCA Dimensionality Reduction
    X_train_pca, X_test_pca = preprocess_with_pca(X_train, X_test)
    
    print(f"PCA completed: Feature dimensions reduced from 784 to {X_train_pca.shape[1]}")

    # Model training and evaluation
    for name, clf in models.items():
        start_time = time.time()
        
        clf.fit(X_train_pca, y_train)
        y_pred = clf.predict(X_test_pca)
        
        acc = accuracy_score(y_test, y_pred)
        cm = confusion_matrix(y_test, y_pred)
        
        results[name].append(acc)
        conf_matrices[name].append(cm)
        
        elapsed = time.time() - start_time
        print(f"[{name}] Accuracy: {acc:.4f} (Time: {elapsed:.2f}s)")

# Summary of results
print("\n" + "="*30)
print("Final Experiment Summary (Mean ± Std)")
print("="*30)

for name, accs in results.items():
    mean_acc = np.mean(accs)
    std_acc = np.std(accs)
    status = "Above Baseline" if mean_acc > 0.9160 else "Below Baseline"
    print(f"{name:18}: {mean_acc:.4f} ± {std_acc:.4f} | {status}")

## 3. Hyperparameter Tuning and Preprocessing Exploration

### 3.1 Import Dependencies and Initialize Logging Dictionaries

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, Binarizer
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np
import joblib

if 'ml_results' not in locals():
    ml_results = {}

best_overall_acc = 0.0
best_overall_model = None

### 3.2 Explore Feature Dimensionality Reduction and SVM Parameter Tuning

In [ ]:
model_name_svm = 'SVM (Tuned + PCA)'
ml_results[model_name_svm] = {}
trial_accs_svm = []

print(f"========== Start Training and Tuning: {model_name_svm} ==========")

# Define Pipeline: Standardization -> PCA -> SVM
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', 'passthrough'),
    ('clf', SVC(random_state=42))
])

# Parameter Grid
param_grid_svm = {
    'pca': ['passthrough', PCA(n_components=0.95), PCA(n_components=0.85)],
    'clf__kernel': ['rbf', 'linear'],
    'clf__C': [0.1, 1, 10]
}

# 2-Trial Protocol
for trial in [1, 2]:
    print(f"\n--- Running Trial {trial} ---")
    
    tr_idx = exp_idx[trial]['train']
    te_idx = exp_idx[trial]['test']
    
    X_train, X_test = X_norm[tr_idx], X_norm[te_idx]
    y_train, y_test = y_all[tr_idx], y_all[te_idx]
    
    # 5-fold cross-validation
    grid_search = GridSearchCV(svm_pipeline, param_grid_svm, cv=5, n_jobs=16, scoring='accuracy')
    grid_search.fit(X_train, y_train)
    
    print(f"Trial {trial} Best Parameters: {grid_search.best_params_}")
    
    # Evaluate on test set
    best_model = grid_search.best_estimator_
    test_acc = accuracy_score(y_test, best_model.predict(X_test))
    print(f"Trial {trial} Test Accuracy: {test_acc:.4f}")
    
    trial_accs_svm.append(test_acc)
    ml_results[model_name_svm][f'exp{trial}'] = test_acc
    
    # Update global best model
    if test_acc > best_overall_acc:
        best_overall_acc = test_acc
        best_overall_model = best_model

# Summary
ml_results[model_name_svm]['mean'] = np.mean(trial_accs_svm)
ml_results[model_name_svm]['std'] = np.std(trial_accs_svm)
print(f"\n>>> {model_name_svm} Final Performance: {ml_results[model_name_svm]['mean']:.4f} ± {ml_results[model_name_svm]['std']:.4f} <<<")

### 3.3 Explore the Impact of Feature Preprocessing on KNN

In [ ]:
model_name_knn = 'KNN (Tuned + Preprocessing)'
ml_results[model_name_knn] = {}
trial_accs_knn = []

print(f"========== Start Training and Tuning: {model_name_knn} ==========")

knn_pipeline = Pipeline([
    ('preprocessor', 'passthrough'), 
    ('clf', KNeighborsClassifier())
])

# Parameter Grid
param_grid_knn = {
    'preprocessor': ['passthrough', MinMaxScaler(), Binarizer(threshold=0.5)], 
    'clf__n_neighbors': [1, 3, 5],
    'clf__weights': ['uniform', 'distance']
}

for trial in [1, 2]:
    print(f"\n--- Running Trial {trial} ---")
    tr_idx = exp_idx[trial]['train']
    te_idx = exp_idx[trial]['test']
    
    X_train, X_test = X_norm[tr_idx], X_norm[te_idx]
    y_train, y_test = y_all[tr_idx], y_all[te_idx]
    
    grid_search = GridSearchCV(knn_pipeline, param_grid_knn, cv=5, n_jobs=16, scoring='accuracy')
    grid_search.fit(X_train, y_train)
    
    print(f"Trial {trial} Best Parameters: {grid_search.best_params_}")
    
    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    print(f"Trial {trial} Test Accuracy: {test_acc:.4f}")
    
    trial_accs_knn.append(test_acc)
    ml_results[model_name_knn][f'exp{trial}'] = test_acc
    
    if test_acc > best_overall_acc:
        best_overall_acc = test_acc
        best_overall_model = best_model
        
    # Plot confusion matrix for Error Analysis in Trial 2
    if trial == 2:
        cm = confusion_matrix(y_test, y_pred)
        disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        fig, ax = plt.subplots(figsize=(6, 6))
        disp.plot(cmap=plt.cm.Blues, ax=ax, colorbar=False)
        plt.title(f"{model_name_knn} Confusion Matrix (Trial 2)")
        plt.show()

ml_results[model_name_knn]['mean'] = np.mean(trial_accs_knn)
ml_results[model_name_knn]['std'] = np.std(trial_accs_knn)
print(f"\n>>> {model_name_knn} Final Performance: {ml_results[model_name_knn]['mean']:.4f} ± {ml_results[model_name_knn]['std']:.4f} <<<")

## 4. HOG Features + Soft Voting Ensemble Model

### 4.1 Extract HOG Image Features

In [ ]:
import numpy as np
import time
from skimage.feature import hog

print("========== Stage 1: Extract HOG Features from Images ==========")

def extract_hog_features(X_data):
    """
    Restore original 784-dimensional vectors to 28x28 images and extract HOG features
    """
    hog_features = []
    for i in range(X_data.shape[0]):
        image = X_data[i].reshape((28, 28))
        
        fd = hog(image, orientations=9, pixels_per_cell=(4, 4),
                 cells_per_block=(2, 2), visualize=False, feature_vector=True)
        hog_features.append(fd)
    return np.array(hog_features)

start_time = time.time()
print(f"Extracting HOG features for {X_norm.shape[0]} images, please wait...")

X_hog_all = extract_hog_features(X_norm) 

print(f"HOG feature extraction completed! Time elapsed: {time.time() - start_time:.2f} seconds.")
print(f"Feature dimension change: Original 784 dims -> HOG {X_hog_all.shape[1]} dims!")

### 4.2 Train Voting Ensemble Model

In [ ]:
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

print("========== Stage 2: Train Voting Ensemble Model ==========")

clf1 = SVC(kernel='rbf', C=5, gamma='scale', probability=True, random_state=42)
clf2 = SVC(kernel='poly', degree=3, C=1, probability=True, random_state=42)
clf3 = KNeighborsClassifier(n_neighbors=3, weights='distance')

ensemble_model = VotingClassifier(
    estimators=[('rbf_svm', clf1), ('poly_svm', clf2), ('knn', clf3)],
    voting='soft',
    n_jobs=16
)

model_name_ultimate = 'HOG + Voting Ensemble'
if 'ml_results' not in locals(): ml_results = {}
ml_results[model_name_ultimate] = {}
trial_accs_ultimate = []

if 'best_ultimate_acc' not in locals(): best_ultimate_acc = 0
if 'best_ultimate_model' not in locals(): best_ultimate_model = None

for trial in [1, 2]:
    print(f"\n--- Running Trial {trial} ---")
    
    tr_idx = exp_idx[trial]['train']
    te_idx = exp_idx[trial]['test']
    
    X_train_hog, X_test_hog = X_hog_all[tr_idx], X_hog_all[te_idx]
    y_train, y_test = y_all[tr_idx], y_all[te_idx]
    
    print("Training ensemble model (may take a few dozen seconds)...")
    ensemble_model.fit(X_train_hog, y_train)
    
    y_pred = ensemble_model.predict(X_test_hog)
    test_acc = accuracy_score(y_test, y_pred)
    print(f"Trial {trial} Test Accuracy: {test_acc:.4f}")
    
    trial_accs_ultimate.append(test_acc)
    ml_results[model_name_ultimate][f'exp{trial}'] = test_acc
    
    if test_acc > best_ultimate_acc:
        best_ultimate_acc = test_acc
        best_ultimate_model = ensemble_model

mean_acc = np.mean(trial_accs_ultimate)
std_acc = np.std(trial_accs_ultimate)
ml_results[model_name_ultimate]['mean'] = mean_acc
ml_results[model_name_ultimate]['std'] = std_acc

print(f"\n>>> {model_name_ultimate} Final Average Performance: {mean_acc:.4f} ± {std_acc:.4f} <<<")

## 5. Comparison of Machine Learning Model Performances

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

print("========== Generating Performance Comparison Chart for All Traditional ML Models ==========")

# Ensure result directory exists
if not os.path.exists('result'):
    os.makedirs('result')

ml_results_to_plot = ml_results.copy()

# Add untuned Baseline model data manually
if '1-NN (Baseline)' not in ml_results_to_plot:
    ml_results_to_plot['1-NN (Baseline)'] = {'mean': 0.9160, 'std': 0.0035}

if 'SVM (Untuned)' not in ml_results_to_plot:
    ml_results_to_plot['SVM (Untuned)'] = {'mean': 0.9410, 'std': 0.0020}

# Sort models by accuracy from lowest to highest
sorted_models = sorted(ml_results_to_plot.keys(), key=lambda x: ml_results_to_plot[x]['mean'])
means = [ml_results_to_plot[m]['mean'] for m in sorted_models]
stds = [ml_results_to_plot[m]['std'] for m in sorted_models]

# Automatically assign colors based on model name
colors = []
for m in sorted_models:
    if 'Baseline' in m or 'Untuned' in m:
        colors.append('lightgray')   # Gray for untuned baselines
    elif 'HOG' in m or 'Ensemble' in m:
        colors.append('gold')        # Gold for advanced models
    else:
        colors.append('steelblue')   # Blue for regular tuned models

fig, ax = plt.subplots(figsize=(12, 6), dpi=100)

bars = ax.bar(sorted_models, means, yerr=stds, capsize=8, 
              color=colors, alpha=0.9, edgecolor='black', linewidth=1.2)

# Add 1-NN baseline line
baseline_acc = 0.9160
ax.axhline(y=baseline_acc, color='red', linestyle='--', linewidth=2, 
           label=f'1-NN Baseline ({baseline_acc:.4f})')

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 0.003, 
            f'{yval:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_title('Evolution of Traditional ML Models (Untuned vs. Tuned vs. HOG)', 
             fontsize=15, fontweight='bold', pad=20)
ax.set_ylabel('Test Accuracy', fontsize=13)
ax.set_xlabel('Models', fontsize=13)

min_acc = min(means)
ax.set_ylim(min_acc - 0.02, 1.0) 

plt.xticks(rotation=15, ha='right')

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='lightgray', edgecolor='black', label='Untuned Baselines'),
    Patch(facecolor='steelblue', edgecolor='black', label='Tuned via GridSearch'),
    Patch(facecolor='gold', edgecolor='black', label='Advanced (HOG + Ensemble)')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11)

ax.grid(True, axis='y', alpha=0.3, linestyle='--')

plt.tight_layout()
save_path = 'result/00_ml_evolution_comparison.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Chart successfully saved to: {save_path}")

plt.show()